In [1]:
import pandas as pd
import 電圧温度換算コード  # 上で作成した換算コードのファイルをインポート
import tkinter as tk
from tkinter import filedialog
import os

def get_target_file():
    """ファイル選択ダイアログを最前面で表示する関数"""
    root = tk.Tk()
    root.withdraw()
    
    root.attributes('-topmost', True) 
    root.lift()
    root.focus_force()

    print("ファイル選択ダイアログを開いています...")

    file_path = filedialog.askopenfilename(
        title="加工対象のデータファイル（CSV）を選択してください",
        filetypes=[("CSVファイル", "*.csv"), ("すべてのファイル", "*.*")]
    )
    
    root.destroy()
    return file_path

def main():
    input_file = get_target_file()

    if not input_file:
        print("ファイルの選択がキャンセルされました。処理を終了します。")
        return

    dir_name = os.path.dirname(input_file)
    base_name = os.path.basename(input_file)
    output_file = os.path.join(dir_name, f"換算結果_{base_name}")

    print(f"読み込みファイル: {input_file}")
    
    try:
        df = pd.read_csv(input_file)
    except Exception as e:
        print(f"ファイルの読み込み中にエラーが発生しました: {e}")
        return

    # 【変更箇所】ターゲットとなる列名を実際のCSVに合わせて指定
    target_col1 = '(1)HA-V05_smoothed'
    target_col2 = '(1)HA-V05_0.7秒前の値(V)'

    if target_col1 not in df.columns or target_col2 not in df.columns:
        print(f"エラー: 選択したデータの中に '{target_col1}' または '{target_col2}' の列が見つかりません。")
        return

    results = []
    print("換算コードにデータを渡し、一括処理を実行中...")

    # データを1行ずつループ処理
    for index, row in df.iterrows():
        # 指定した列からデータを取得
        data_Vom = row[target_col1]
        data_Vos = row[target_col2]
        
        # 換算コード内の関数を呼び出し、計算を実行
        final_value = 電圧温度換算コード.execute_main_calc(data_Vom, data_Vos)
        
        results.append(final_value)

    # 処理結果を元のデータフレームに「換算結果」という新しい列として追加
    df['最高値換算結果(℃)'] = results

    # 保存前にVS Code上にプレビューを出力
    print("\n▼▼▼ 換算結果のプレビュー ▼▼▼")
    print(df)
    print("▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲\n")

    # 新しいCSVファイルとして出力
    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    
    print(f"すべての処理が完了しました！\n結果を保存しました:\n{output_file}")

if __name__ == "__main__":
    main()

ファイル選択ダイアログを開いています...
読み込みファイル: C:/Users/0uh2j/OneDrive - 学校法人 日本工業大学 (1)/えーあい修士/学会発表_成形加工学会2026/Pythonグラフ資料/温度圧縮データ_1万行まで/035_210℃_080_1.0mm_smoothed_10k.csv
換算コードにデータを渡し、一括処理を実行中...

▼▼▼ 換算結果のプレビュー ▼▼▼
       Time (s)  (1)HA-V05_smoothed  (1)HA-V05_0.7秒前の値(V)  最高値換算結果(℃)
0        0.0000            1.427179              1.407876   51.168014
1        0.0004            1.427354              1.407876   51.246541
2        0.0008            1.427708              1.407876   51.404570
3        0.0012            1.427768              1.407876   51.431167
4        0.0016            1.427758              1.407876   51.426737
...         ...                 ...                   ...         ...
9996     3.9984            1.877571              1.407876  109.328927
9997     3.9988            1.877544              1.407876  109.327157
9998     3.9992            1.877277              1.407876  109.309940
9999     3.9996            1.872150              1.407876  108.977985
10000    4.0000           